# 08. Mini Project: 통합 에이전트 시스템 구현

1-3일차 전체 내용 통합

---

### 프로젝트 목표

**SKILL.md 기반 하네스**로 W1(엑셀 Agent) + W2(SQL Agent) + W3(RAG Agent)를 **SubAgent로 통합**하여  
자연어 질문에 적절한 Agent로 라우팅하는 시스템을 구축한다.

### 아키텍처

```
사용자 질문 → Supervisor → SubAgent Harness → {엑셀 SubAgent, SQL SubAgent, RAG SubAgent} → 통합 응답
                              ↑
                         SKILL.md가 각 SubAgent의 행동을 선언적으로 정의
```

### SKILL.md 기반 하네스란?

| 구성 요소 | 역할 |
|-----------|------|
| **SKILL.md** | 각 Agent의 행동·도구·역할을 선언적으로 정의하는 파일 |
| **SubAgent** | SKILL.md를 system_prompt로 로드하여 독립된 컨텍스트에서 실행 |
| **Supervisor** | SubAgent들을 오케스트레이션하는 상위 Agent |

### 데이터셋 선택

| 세트 | 경로 | 도메인 |
|------|------|--------|
| **Set A** | `sample_data/pm/` | 프로젝트 관리 |
| **Set B** | `sample_data/semi/` | 반도체 공정 |

> 아래 구현은 **Set A(프로젝트 관리)** 기준으로 작성되어 있습니다.  
> Set B로 전환하려면 `DATA_SET = "semi"` 로 변경하고 파일명·컬럼명을 조정하세요.



## 1) 환경 설정 & 데이터 준비
> 진행: [▓░░░░░░] 1/7


In [1]:
# [P-1] : 라이브러리 설치
# [핵심] 멀티에이전트 통합에 필요한 전체 패키지
# [주의] deepagents 가 핵심 — Supervisor + SubAgent 구현체
#        notebook 환경에서는 !uv 대신 %pip 매직을 써야 현재 커널에 정확히 설치됩니다.

%pip install -q langchain langchain-core langchain-openai langgraph deepagents \
    langchain-text-splitters \
    pandas openpyxl python-dotenv tabulate pydantic

Note: you may need to restart the kernel to use updated packages.


c:\Users\pkw80\hynix-test\.venv\Scripts\python.exe: No module named pip


In [2]:
# [P-2] : 환경변수 로드 및 API 키 검증
# [핵심] load_dotenv()로 .env 파일의 키-값 쌍을 환경변수로 로드
# [주의] .env 파일이 없으면 API 호출 실패
import os
from dotenv import load_dotenv

load_dotenv(override=True)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
assert OPENAI_API_KEY, "OPENAI_API_KEY가 .env에 설정되어 있지 않습니다."
print(f"API 키 로드 완료: {OPENAI_API_KEY[:8]}...")

API 키 로드 완료: sk-proj-...


In [3]:
# [P-4] : LLM 및 임베딩 모델 초기화
# [핵심] 모든 Agent가 동일한 llm 인스턴스를 공유 — 비용 예측 용이
# [주의] temperature=0 → 라우팅 결정이 결정적이어야 함
#        model 이름은 본인 계정에서 사용 가능한 값으로 바꿔도 됩니다.

from langchain_openai import ChatOpenAI, OpenAIEmbeddings

llm = ChatOpenAI(model="gpt-5.4-mini", temperature=0)
embedding = OpenAIEmbeddings(model="text-embedding-3-small")

print(f"LLM: {llm.model_name}")
print(f"Embedding: {embedding.model}")

LLM: gpt-5.4-mini
Embedding: text-embedding-3-small


In [4]:
# [P-5] : 데이터셋 선택
# [핵심] DATA_SET 변수 하나로 Set A / Set B 전환
# [주의] 파일명·컬럼명이 달라지므로 이후 코드도 함께 확인

DATA_SET = "pm"          # "pm" 또는 "semi"
DATA_DIR = f"sample_data/{DATA_SET}"

print(f"데이터셋: {DATA_SET} ({DATA_DIR})")

데이터셋: pm (sample_data/pm)


## 2) 개별 Agent 재활용 (W1~W3 코드 통합)
> 진행: [▓▓░░░░░] 2/7


### 2-1. 엑셀 Agent (W1)

In [5]:
# [P-6] : 엑셀 데이터 로드
# [핵심] W1 워크숍에서 작성한 pandas Tool 코드를 그대로 가져옴
# [주의] 파일 경로가 다를 경우 DATA_DIR을 앞서 설정한 변수로 사용

import pandas as pd
from langchain_core.tools import tool

EXCEL_PATH = f"{DATA_DIR}/pm_projects.xlsx"

df_projects  = pd.read_excel(EXCEL_PATH, sheet_name="프로젝트현황")
df_monthly   = pd.read_excel(EXCEL_PATH, sheet_name="월별실적")
df_resources = pd.read_excel(EXCEL_PATH, sheet_name="리소스배분")

print(f"프로젝트현황: {df_projects.shape}")
print(f"월별실적: {df_monthly.shape}")
print(f"리소스배분: {df_resources.shape}")

프로젝트현황: (30, 11)
월별실적: (120, 6)
리소스배분: (50, 5)


In [6]:
# [P-7] : 엑셀 분석 Tool 정의
# [핵심] W1에서 구현한 Tool을 그대로 복사하거나 여기에 재정의
# [주의] Tool docstring은 Supervisor 라우팅의 핵심 — 정확히 작성

@tool
def get_project_summary(status: str = None) -> str:
    """프로젝트 현황을 요약합니다. status 필터 가능 (예: '진행중', '지연', '완료', '보류')."""
    df = df_projects.copy()
    if status:
        df = df[df["status"] == status]
    summary = df.groupby("status").agg(
        count=("project_id", "count"),
        total_budget=("budget_million", "sum"),
        avg_progress=("progress_pct", "mean"),
    ).reset_index()
    return summary.to_string(index=False)


@tool
def analyze_monthly_performance(month: str = None) -> str:
    """월별 실적(완료 태스크, 이슈 건수)을 분석합니다. month 필터 가능 (형식: 'YYYY-MM')."""
    df = df_monthly.copy()
    if month:
        df = df[df["month"] == month]
    result = df.groupby("month").agg(
        completed_tasks=("completed_tasks", "sum"),
        total_issues=("issues_count", "sum"),
    ).reset_index()
    return result.to_string(index=False)


@tool
def get_resource_allocation(
    project_id: str = None,
    role: str = None,
    member_name: str = None,
) -> str:
    """리소스 배분 현황을 조회합니다. project_id, role, member_name 으로 필터할 수 있습니다."""
    df = df_resources.copy()
    if project_id:
        df = df[df["project_id"] == project_id]
    if role:
        df = df[df["role"] == role]
    if member_name:
        df = df[df["member_name"] == member_name]
    if df.empty:
        return "조건에 맞는 리소스 배분 데이터가 없습니다."
    return df.to_string(index=False)


excel_tools = [get_project_summary, analyze_monthly_performance, get_resource_allocation]
print(f"엑셀 Tool 등록: {[t.name for t in excel_tools]}")

엑셀 Tool 등록: ['get_project_summary', 'analyze_monthly_performance', 'get_resource_allocation']


In [7]:
# [P-8] : 엑셀 Agent 생성
# [핵심] create_agent(model, tools, system_prompt, name) — name으로 Supervisor가 라우팅
# [주의] name="excel_agent"가 Supervisor 프롬프트의 라우팅 규칙과 일치해야 함
from langchain.agents import create_agent

excel_agent = create_agent(
    model=llm,
    tools=excel_tools,
    system_prompt=(
        "당신은 프로젝트 관리 엑셀 데이터 분석 전문가입니다. "
        "pandas 기반 Tool을 활용하여 정확한 데이터 기반 답변을 제공하세요."
    ),
    name="excel_agent",
)

print("엑셀 Agent 생성 완료")

엑셀 Agent 생성 완료


### 2-2. SQL Agent (W2)

In [8]:
# [P-9] : SQLite DB 연결
# [핵심] W2 워크숍에서 사용한 DB 파일을 그대로 활용
# [주의] DB 파일 경로가 정확해야 합니다

import sqlite3

DB_PATH = f"{DATA_DIR}/pm.db"
conn = sqlite3.connect(DB_PATH, check_same_thread=False)

tables = conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()
print(f"테이블 목록: {[t[0] for t in tables]}")

테이블 목록: ['members', 'tasks', 'issues', 'meetings']


In [9]:
# [P-10] : SQL Tool 정의
# [핵심] W2에서 구현한 SQL Tool을 여기에 재정의
# [주의] SELECT 전용 제약을 system_prompt에 명시

@tool
def run_sql_query(query: str) -> str:
    """SQLite DB에서 SQL 쿼리를 실행하고 결과를 반환합니다. 프로젝트·태스크·리소스 정보를 조회할 때 사용하세요."""
    try:
        df = pd.read_sql_query(query, conn)
        return df.to_string(index=False)
    except Exception as e:
        return f"SQL 오류: {e}"


@tool
def get_db_schema() -> str:
    """DB 스키마(테이블명·컬럼명)를 반환합니다. SQL 작성 전에 먼저 호출하세요."""
    schema_info = []
    for table_name, in conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall():
        cols = conn.execute(f"PRAGMA table_info({table_name})").fetchall()
        col_names = [c[1] for c in cols]
        schema_info.append(f"{table_name}: {', '.join(col_names)}")
    return "\n".join(schema_info)


sql_tools = [run_sql_query, get_db_schema]
print(f"SQL Tool 등록: {[t.name for t in sql_tools]}")

SQL Tool 등록: ['run_sql_query', 'get_db_schema']


In [10]:
# [P-11] : SQL Agent 생성
# [핵심] name="sql_agent" — Supervisor 라우팅에 사용
# [주의] system_prompt에 "스키마 먼저 확인" 지시가 없으면 잘못된 SQL 생성 가능
sql_agent = create_agent(
    model=llm,
    tools=sql_tools,
    system_prompt=(
        "당신은 프로젝트 관리 DB 전문가입니다. "
        "SQL Tool을 사용하여 정확한 데이터를 조회하고 답변하세요. "
        "SQL 작성 전에 먼저 get_db_schema로 스키마를 확인하세요."
    ),
    name="sql_agent",
)

print("SQL Agent 생성 완료")

SQL Agent 생성 완료


### 2-3. RAG Agent (W3)

In [11]:
# [P-12] : 인메모리 벡터스토어 기반 문서 검색 도구 구성
# [핵심] 마크다운 문서를 로드 → 분할 → 임베딩 → InMemoryVectorStore에 적재한다.
#        (Chroma 대신 메모리 벡터스토어 사용 — 별도 DB 파일/디렉터리가 필요 없음)
# [주의] 인메모리이므로 커널을 재시작하면 매번 다시 임베딩한다. k 값을 너무 키우면 잡음이 섞인다.

import os, glob
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.tools import tool

DOCS_DIR = f"{DATA_DIR}/{DATA_SET}_docs"


def build_vectorstore(docs_dir):
    """docs_dir 안의 .md 파일을 읽어 InMemoryVectorStore를 구축한다."""
    docs = []
    for path in sorted(glob.glob(os.path.join(docs_dir, "*.md"))):
        with open(path, encoding="utf-8") as f:
            docs.append(Document(page_content=f.read(),
                                 metadata={"source": os.path.basename(path)}))
    splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
    chunks = splitter.split_documents(docs)
    return InMemoryVectorStore.from_documents(chunks, embedding)


vectorstore = build_vectorstore(DOCS_DIR)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})


@tool
def search_documents(query: str) -> str:
    """벡터 스토어에서 프로젝트 문서를 검색합니다. 절차·정책·가이드라인 관련 질문에 사용하세요."""
    docs = retriever.invoke(query)
    if not docs:
        return "관련 문서를 찾을 수 없습니다."
    return "\n\n".join(
        f"[출처: {d.metadata.get('source', 'N/A')}]\n{d.page_content}"
        for d in docs
    )


rag_tools = [search_documents]
print(f"RAG Tool 등록 완료: {[t.name for t in rag_tools]}")

RAG Tool 등록 완료: ['search_documents']


In [12]:
# [P-13] : RAG Agent 생성
# [핵심] name="rag_agent" — Supervisor 라우팅에 사용
# [주의] Tool docstring이 부정확하면 Supervisor가 이 Agent를 선택하지 않음
rag_agent = create_agent(
    model=llm,
    tools=rag_tools,
    system_prompt=(
        "당신은 프로젝트 관리 문서 전문가입니다. "
        "search_documents Tool을 사용하여 절차·정책·가이드라인 관련 질문에 답변하세요."
    ),
    name="rag_agent",
)

print("RAG Agent 생성 완료")

RAG Agent 생성 완료


## 3) SKILL.md 기반 하네스 설계
> 진행: [▓▓▓░░░░] 3/7

각 워크숍에서 만든 Agent를 **SKILL.md로 선언**합니다.  
SKILL.md는 Agent의 행동을 **선언적으로 정의**하는 파일로, SubAgent의 `system_prompt`로 로드됩니다.

### SKILL.md 구조 복습

```yaml
---
name: agent-name           # 스킬 식별자 (소문자+하이픈)
description: >             # 설명 — SubAgent 매칭에 사용
  이 Agent가 수행하는 역할을 기술합니다.
allowed-tools: tool1 tool2 # 사용 가능한 도구 목록
---

# Agent 행동 지침 (본문)
여기에 Agent의 상세 행동 규칙을 작성합니다.
```

### 핵심 포인트

| 항목 | 설명 |
|------|------|
| **Progressive Disclosure** | 프론트매터(이름, 설명)만 먼저 로드 → 필요 시 전체 로드 |
| **토큰 효율** | 대용량 지침도 필요할 때만 로드하여 컨텍스트 절약 |
| **SubAgent 연동** | SKILL.md 본문을 `system_prompt`로 주입하여 행동 가이드 |


In [13]:
# [P-14a] : 각 Agent의 SKILL.md 정의
# [핵심] SKILL.md 본문이 SubAgent의 system_prompt가 됨
# [주의] description은 Supervisor가 라우팅 판단에 사용하므로 명확히 작성
#        allowed-tools 에는 실제 등록된 Tool 이름을 적는다.

# --- 엑셀 분석 Agent SKILL.md ---
EXCEL_SKILL = """\
---
name: excel-analysis-agent
description: 엑셀 데이터를 분석하는 Agent
allowed-tools: get_project_summary analyze_monthly_performance get_resource_allocation
---

# 엑셀 분석 Agent

## 역할
엑셀(프로젝트현황·월별실적·리소스배분) 데이터를 pandas Tool로 분석합니다.

## 행동 규칙
- 예산, 진행률, 월별 실적, 리소스 배분 관련 질문을 처리
- 결과는 숫자와 근거를 함께 제시
"""

# --- SQL 조회 Agent SKILL.md ---
SQL_SKILL = """\
---
name: sql-query-agent
description: SQL 데이터베이스를 조회하여 태스크·리소스 현황을 분석하는 Agent
allowed-tools: run_sql_query get_db_schema
---

# SQL 조회 Agent

## 역할
SQLite 데이터베이스에서 프로젝트 관리 데이터를 조회합니다.

## 행동 규칙
- SQL 작성 전 반드시 get_db_schema로 스키마 확인
- SELECT 전용 — INSERT/UPDATE/DELETE 금지
- 태스크 목록, 이슈, 회의록, 멤버 조회를 처리
"""

# --- RAG 문서검색 Agent SKILL.md ---
RAG_SKILL = """\
---
name: rag-document-agent
description: 벡터 스토어에서 문서를 검색하여 절차·정책·가이드라인 질문에 답변하는 Agent
allowed-tools: search_documents
---

# RAG 문서검색 Agent

## 역할
인메모리 벡터 스토어에서 관련 문서를 검색하고 답변합니다.

## 행동 규칙
- 절차, 정책, 리스크 관리, 가이드라인 질문을 처리
- 검색 결과의 출처를 반드시 명시
- 문서에 없는 내용은 "문서에서 확인되지 않음"으로 응답
"""

print("SKILL.md 3개 정의 완료")
print(f"  - excel-analysis-agent: {len(EXCEL_SKILL)} chars")
print(f"  - sql-query-agent: {len(SQL_SKILL)} chars")
print(f"  - rag-document-agent: {len(RAG_SKILL)} chars")

SKILL.md 3개 정의 완료
  - excel-analysis-agent: 288 chars
  - sql-query-agent: 299 chars
  - rag-document-agent: 275 chars


In [14]:
# [P-14b] : SKILL.md를 SubAgent로 등록
# [핵심] SubAgent dict의 system_prompt에 SKILL.md 본문을 주입
# [주의] description은 Supervisor가 호출 판단에 사용 — SKILL.md 프론트매터와 일치시킬 것

excel_subagent = {
    "name": "excel-agent",
    "description": "엑셀 데이터 분석 전문 Agent — 예산, 진행률, 월별 실적, 리소스 분석",
    "system_prompt": EXCEL_SKILL,
    "tools": excel_tools,
}

sql_subagent = {
    "name": "sql-agent",
    "description": "SQL 데이터 조회 전문 Agent — 태스크 목록, 이슈, 회의록, DB 질의",
    "system_prompt": SQL_SKILL,
    "tools": sql_tools,
}

rag_subagent = {
    "name": "rag-agent",
    "description": "문서 검색 및 분석 전문 Agent — 절차, 정책, 가이드라인 검색",
    "system_prompt": RAG_SKILL,
    "tools": rag_tools,
}

subagents = [excel_subagent, sql_subagent, rag_subagent]

print("SubAgent 등록 완료:")
for sa in subagents:
    print(f"  - {sa['name']}: {sa['description'][:40]}...")

SubAgent 등록 완료:
  - excel-agent: 엑셀 데이터 분석 전문 Agent — 예산, 진행률, 월별 실적, 리소스...
  - sql-agent: SQL 데이터 조회 전문 Agent — 태스크 목록, 이슈, 회의록, D...
  - rag-agent: 문서 검색 및 분석 전문 Agent — 절차, 정책, 가이드라인 검색...


### SubAgent vs 기존 create_agent 비교

| 항목 | create_agent (기존) | SubAgent + SKILL.md (신규) |
|------|-------------------|---------------------------|
| **행동 정의** | system_prompt 문자열 직접 작성 | SKILL.md 파일로 선언적 관리 |
| **컨텍스트** | 메인 Agent와 공유 (블로트 위험) | 독립된 컨텍스트 (격리) |
| **결과 전달** | 전체 메시지 히스토리 공유 | 요약만 메인 Agent에 반환 |
| **재사용성** | 코드에 종속 | SKILL.md 파일로 이식 가능 |
| **확장** | 코드 수정 필요 | SKILL.md 추가만으로 Agent 확장 |

> 기존 `create_agent` 코드(P-8, P-11, P-13)는 **Tool 정의와 테스트 용도**로 유지합니다.  
> 실제 Supervisor 통합은 아래 **SubAgent 기반**으로 진행합니다.


## 4) Supervisor + SubAgent 오케스트레이션
> 진행: [▓▓▓▓░░░] 4/7


In [15]:
# [P-14c] : Supervisor + SubAgent 통합 구성
# [핵심] create_deep_agent로 여러 SubAgent를 묶는 Supervisor를 구성한다
# [주의] subagents 목록의 이름과 역할이 system_prompt의 라우팅 기준과 일치해야 한다

from deepagents import create_deep_agent

SUPERVISOR_PROMPT = """당신은 통합 어시스턴트입니다. 사용자의 질문을 분석하여 적절한 서브에이전트에게 위임하세요.

라우팅 기준:
- 엑셀 데이터 분석 (예산, 진행률, 월별 실적) → excel-agent
- DB 조회 (태스크 목록, 리소스 현황, SQL 질의) → sql-agent
- 문서 검색 (절차, 정책, 리스크 관리, 가이드라인) → rag-agent
- 복합 질문 → 여러 서브에이전트를 순차 호출하여 종합 답변 생성

서브에이전트의 결과를 종합하여 최종 답변을 한국어로 작성하세요."""

app = create_deep_agent(
    model=llm,
    system_prompt=SUPERVISOR_PROMPT,
    subagents=subagents,
)

print("SubAgent 기반 Supervisor 구성 완료")
print(f"등록된 SubAgent: {[sa['name'] for sa in subagents]}")


SubAgent 기반 Supervisor 구성 완료
등록된 SubAgent: ['excel-agent', 'sql-agent', 'rag-agent']


## 5) 통합 테스트 & 디버깅
> 진행: [▓▓▓▓▓░░] 5/7


In [16]:
# [P-15] : 통합 질문 실행 및 결과 출력 헬퍼
# [핵심] stream()으로 각 SubAgent의 중간 응답을 확인할 수 있다
# [주의] verbose=True일 때만 상세 로그를 출력한다

from langgraph.types import Overwrite


def _normalize_messages(raw_messages):
    if isinstance(raw_messages, Overwrite):
        raw_messages = raw_messages.value
    if raw_messages is None:
        return []
    if isinstance(raw_messages, (list, tuple)):
        return list(raw_messages)
    return [raw_messages]


def run_and_print(question: str, verbose: bool = True):
    """질문을 실행하고 스트리밍으로 결과를 출력한다."""
    print(f"\n{'=' * 60}")
    print(f"질문: {question}")
    print('=' * 60)

    final = None
    for chunk in app.stream({"messages": [{"role": "user", "content": question}]}):
        if verbose:
            for agent_name, state in chunk.items():
                if not hasattr(state, 'get'):
                    continue
                for m in _normalize_messages(state.get("messages")):
                    content = getattr(m, "content", None)
                    if content:
                        role = type(m).__name__
                        print(f"  [{agent_name}/{role}] {str(content)[:200]}")
        final = chunk

    if final is None:
        print("\n최종 답변을 받지 못했습니다.")
        return None

    for state in final.values():
        if not hasattr(state, 'get'):
            continue
        msgs = _normalize_messages(state.get("messages"))
        if msgs:
            content = getattr(msgs[-1], "content", msgs[-1])
            print(f"\n최종 답변:\n{content}")
    return final

In [17]:
# [P-16] : 테스트 1 — 엑셀 Agent 라우팅
# [핵심] Supervisor가 적절한 Agent로 라우팅하는지 확인
# [주의] 기대: supervisor → excel_agent — 라우팅 실패 시 Supervisor 프롬프트 점검

run_and_print("지연 중인 프로젝트의 예산 현황을 분석해줘", verbose=True)


질문: 지연 중인 프로젝트의 예산 현황을 분석해줘
  [tools/ToolMessage] DB/SQL 관점에서 확인한 결과, **현재 스키마에는 예산 관련 테이블이 없습니다.**  
확인된 테이블은 `members`, `tasks`, `issues`, `meetings`뿐이며, `budget`, `cost`, `expense`, `project` 같은 예산/프로젝트 마스터 테이블은 보이지 않습니다.  
따라서 **지연 프로젝트의 예산 현황(총예
  [model/AIMessage] 현재 DB 기준으로는 **지연 프로젝트의 예산 현황을 직접 분석할 수 없습니다.**

확인된 이유:
- 예산 관련 테이블이 없음: `budget`, `cost`, `expense` 등
- 프로젝트 마스터 테이블이 없음: 프로젝트명/예산/집행액 연결 불가
- 현재는 `tasks`, `issues`, `meetings`, `members` 정도만 확인됨

다만


{'TodoListMiddleware.after_model': None}

In [18]:
# [P-17] : 테스트 2 — SQL Agent 라우팅
# [핵심] Supervisor가 적절한 Agent로 라우팅하는지 확인
# [주의] 기대: supervisor → sql_agent — 라우팅 실패 시 Supervisor 프롬프트 점검

run_and_print("현재 보류(Blocked) 상태인 태스크 목록을 보여줘", verbose=True)


질문: 현재 보류(Blocked) 상태인 태스크 목록을 보여줘
  [tools/ToolMessage] 현재 DB 기준으로 **상태가 `보류`인 태스크**는 확인되지만, **블로커 사유/메모와 최근 업데이트 시각 컬럼은 스키마에 없어 조회할 수 없습니다.**  
대신 확인 가능한 핵심 정보만 정리했습니다.

| 태스크 ID | 제목/요약 | 상태 | 담당자 | 우선순위 | 생성일 | 마감일 |
|---|---|---|---|---|---|---|
| TSK-
  [model/AIMessage] 현재 DB 기준으로 **보류(Blocked)** 상태인 태스크는 **총 39건**입니다.

예시로 일부만 보면:
- TSK-0080 — DB 스키마 설계 — 정건우
- TSK-0010 — 운영 이관 — 정건우
- TSK-0022 — 문서화 — 이서연
- TSK-0090 — DB 스키마 설계 — 윤아름
- TSK-0175 — DB 스키마 설계 — 김민준
- 


{'TodoListMiddleware.after_model': None}

In [19]:
# [P-18] : 테스트 3 — RAG Agent 라우팅
# [핵심] Supervisor가 적절한 Agent로 라우팅하는지 확인
# [주의] 기대: supervisor → rag_agent — 라우팅 실패 시 Supervisor 프롬프트 점검

run_and_print("리스크 관리 절차를 설명해줘", verbose=True)


질문: 리스크 관리 절차를 설명해줘
  [tools/ToolMessage] 아래는 문서 검색 결과를 바탕으로 정리한 **리스크 관리 절차 요약**입니다.  
**근거 문서:** `리스크_관리_절차서.md`

### 1) 핵심 절차
- **식별(Identify)**: 브레인스토밍, 체크리스트 검토, 인터뷰, 가정 분석으로 리스크를 발굴
- **분석(Analyze)**: 가능성·영향도를 기준으로 정성/정량 평가
- **계획 수립(Pl
  [model/AIMessage] 리스크 관리 절차는 보통 아래 흐름으로 진행합니다.

1. **식별(Identify)**  
   브레인스토밍, 체크리스트, 인터뷰, 가정 분석 등을 통해 잠재 리스크를 찾습니다.

2. **분석(Analyze)**  
   각 리스크의 **발생 가능성**과 **영향도**를 기준으로 정성적/정량적으로 평가합니다.

3. **대응계획 수립(Plan Respo


{'TodoListMiddleware.after_model': None}

In [20]:
# [P-19] : 테스트 4 — 복합 질문 (멀티 에이전트)
# [핵심] system_prompt 튜닝으로 라우팅 정확도 향상
# [주의] 기대: supervisor → excel_agent + sql_agent + rag_agent 순차 호출

run_and_print(
    "프로젝트 PRJ-001의 예산 현황(엑셀), 태스크 현황(DB), 관련 절차(문서)를 종합 분석해줘",
    verbose=True,
)


질문: 프로젝트 PRJ-001의 예산 현황(엑셀), 태스크 현황(DB), 관련 절차(문서)를 종합 분석해줘
  [tools/ToolMessage] PRJ-001의 예산 현황을 확인하려 했지만, 현재 사용 가능한 엑셀/분석 도구에서는 **PRJ-001의 직접적인 예산 데이터(총예산, 집행액, 월별 예산/집행)** 가 조회되지 않았습니다.  
대신 조회 가능한 범위에서 **프로젝트 요약/월별 실적/리소스 배분** 정보를 확인했고, 그 결과를 아래처럼 정리합니다.

## 1) 총예산 / 집행액 / 잔액 /
  [tools/ToolMessage] PRJ-001에 연결된 태스크만 기준으로 DB를 조회해 정리했습니다.

## 1) 요약
- **총 태스크 수:** 6건
- **진행 중:** 1건
- **완료:** 2건
- **지연:** 4건
- **미착수/보류:** 3건  
  - 할일 1건
  - 보류 2건

> 참고: 지연 여부는 `마감일(due_date) < 오늘` 이고 `완료일(completed_
  [tools/ToolMessage] 아래는 **PRJ-001 관련 절차/정책/가이드라인 문서**를 우선적으로 찾아 요약한 내용입니다.  
※ 검색 결과 기준으로 **PRJ-001이 문서명에 직접 등장하지는 않았고**, 프로젝트 관리 표준 문서들 중 PRJ-001에 적용 가능한 문서를 중심으로 확인했습니다. 문서에서 PRJ-001 전용 규정은 **확인되지 않음**입니다.

---

## 1) 
  [model/AIMessage] 프로젝트 **PRJ-001**에 대해 엑셀 예산, DB 태스크, 문서 절차를 종합하면 아래와 같습니다.

## 1) 예산 현황(엑셀)
- 현재 조회된 결과에서는 **PRJ-001의 직접 예산 수치(총예산/집행액/잔액/집행률)** 는 확인되지 않았습니다.
- 다만 운영 실적 관점에서 보면:
  - **2025-03 완료 태스크 급감**
  - **2025-0


{'TodoListMiddleware.after_model': None}

## 6) 프롬프트 튜닝 & 품질 개선
> 진행: [▓▓▓▓▓▓░] 6/7


In [21]:
# [P-20] : 테스트 케이스 일괄 실행 & 품질 확인
# [핵심] 여러 도메인 질문으로 시스템 견고성 검증
# [주의] 라우팅 오류 발생 시 P-21에서 프롬프트 튜닝 후 재실행

test_questions = [
    ("지연 중인 프로젝트의 예산 현황을 분석해줘",        "excel_agent"),
    ("현재 보류(Blocked) 태스크 목록을 보여줘",               "sql_agent"),
    ("리스크 관리 절차를 설명해줘",                     "rag_agent"),
    ("PRJ-001의 예산, 태스크 현황, 관련 절차를 종합 분석해줘", "multi"),
]

print("=== 라우팅 정확도 테스트 ===\n")
for q, expected in test_questions:
    result = app.invoke({"messages": [{"role": "user", "content": q}]})
    answer = result["messages"][-1].content
    print(f"[기대: {expected}]")
    print(f"Q: {q}")
    print(f"A: {answer[:200]}...")
    print("-" * 60)

=== 라우팅 정확도 테스트 ===

[기대: excel_agent]
Q: 지연 중인 프로젝트의 예산 현황을 분석해줘
A: 지연 중인 프로젝트는 현재 **5개**로 확인됐고, 이들의 **총 예산은 1,260**, **평균 진행률은 38.2%**입니다.

### 분석 요약
- **지연 기준**: 상태가 `지연`인 프로젝트
- **예산 현황**: 총 예산 1,260
- **진행 현황**: 평균 진행률 38.2%로 낮은 편
- **해석**: 예산 초과보다는 **미집행 및 일정 지연 ...
------------------------------------------------------------
[기대: sql_agent]
Q: 현재 보류(Blocked) 태스크 목록을 보여줘
A: 현재 보류(Blocked) 태스크는 다음 2건입니다.

- **TSK-0085** | 문서화 | 담당자: 박지호 | 마지막 업데이트: 2025-04-12
- **TSK-0007** | 사용자 교육 | 담당자: 조진우 | 마지막 업데이트: 2025-04-11

참고:
- DB에는 `blocked` 사유 컬럼이 없어 블록 원인은 확인되지 않았습니다....
------------------------------------------------------------
[기대: rag_agent]
Q: 리스크 관리 절차를 설명해줘
A: 리스크 관리 절차는 일반적으로 다음 순서로 진행됩니다.

1. **리스크 식별**  
2. **리스크 분석**  
3. **대응 계획 수립**  
4. **대응 실행**  
5. **모니터링 및 재검토**  
6. **에스컬레이션/보고**

간단히 설명하면:

- **리스크 식별**: 프로젝트에 영향을 줄 수 있는 위험 요소를 찾아냅니다.
- **리스크 분...
------------------------------------------------------------
[기대: multi]
Q: PRJ-001의 예산, 태스크 현황, 관련 절차를 종합 분석해줘
A: P

In [22]:
# [P-21] : Supervisor 프롬프트 튜닝 (선택)
# [핵심] 라우팅 오류가 발생하면 프롬프트를 수정하여 재컴파일
# [주의] 변경 후 반드시 위 테스트를 다시 실행하여 개선 여부 확인

IMPROVED_PROMPT = """당신은 통합 어시스턴트입니다.

[서브에이전트 선택 규칙]
1. 질문에 '예산', '진행률', '월별 실적', '집행률' → excel-agent
2. 질문에 '태스크', '목록', '조회', 'Blocked', 'DB' → sql-agent
3. 질문에 '절차', '정책', '가이드', '리스크 관리' → rag-agent
4. 여러 도메인이 혼합된 경우 → 각 서브에이전트를 순차 호출하여 결과를 종합

서브에이전트의 결과를 종합하여 최종 답변을 한국어로 작성하세요."""

# 재컴파일 (필요 시 주석 해제)
# app2 = create_deep_agent(
#     model=llm,
#     system_prompt=IMPROVED_PROMPT,
#     subagents=subagents,
# )
# run_and_print("지연 프로젝트 예산 분석", verbose=False)

print("프롬프트 튜닝 셀 준비 완료 — 필요 시 주석 해제 후 실행")


프롬프트 튜닝 셀 준비 완료 — 필요 시 주석 해제 후 실행


## 7) 시연 준비 & 최종 점검
> 진행: [▓▓▓▓▓▓▓] 7/7


In [23]:
# [P-22] : 시연용 질문 일괄 실행
# [핵심] 발표 시 실시간 실행할 대표 질문 목록
# [주의] 시연 질문은 각 Agent가 골고루 호출되도록 구성

demo_questions = [
    "지연 중인 프로젝트의 예산 현황을 분석해줘",
    "현재 보류(Blocked) 태스크 목록을 보여줘",
    "리스크 관리 절차를 설명해줘",
    "프로젝트 PRJ-001의 예산, 태스크 현황, 관련 절차를 종합 분석해줘",
]

for q in demo_questions:
    result = app.invoke({"messages": [{"role": "user", "content": q}]})
    answer = result["messages"][-1].content
    print(f"\nQ: {q}")
    print(f"A: {answer[:300]}...")
    print("-" * 70)


Q: 지연 중인 프로젝트의 예산 현황을 분석해줘
A: 현재 DB 기준으로는 **지연 중인 프로젝트의 예산 현황을 실제 수치로 분석할 수 없습니다.**

확인 결과:
- 프로젝트 테이블이 없어 **지연 프로젝트 목록**을 직접 조회할 수 없음
- 예산 관련 테이블/필드가 없어 **총예산, 집행액, 잔여액**을 조회할 수 없음
- `tasks`, `issues`는 있어 **지연 징후를 간접 추정**할 수는 있지만, 예산 분석에는 부족함

즉, 지금 가능한 결론은:
- **예산 현황 분석 불가**
- **추가 데이터 필요**
  - 프로젝트 기본 정보: `project_id`, `proje...
----------------------------------------------------------------------

Q: 현재 보류(Blocked) 태스크 목록을 보여줘
A: 현재 보류(Blocked) 상태의 태스크는 조회 결과가 없습니다....
----------------------------------------------------------------------

Q: 리스크 관리 절차를 설명해줘
A: 리스크 관리 절차는 보통 다음 순서로 진행합니다.

1. **리스크 식별**
   - 브레인스토밍, 체크리스트, 인터뷰, 가정 분석 등을 통해 잠재 리스크를 찾습니다.

2. **리스크 분석**
   - 발생 가능성과 영향도를 평가해 높음/중간/낮음 등으로 등급을 매깁니다.

3. **대응 계획 수립**
   - 위협 리스크는 회피·전이·완화·수용 전략을 선택하고,
   - 기회 리스크는 활용·강화·공유 전략을 검토합니다.

4. **대응 실행**
   - 정해진 대응 계획을 실제 업무에 반영하고 리스크 로그를 갱신합니다.

5. ...
----------------------------------------------------------------------

Q: 프로젝트 PRJ-001의 예산, 태스크 현황, 관련 절차를 종합 분석해줘
A: PRJ

In [24]:
# [P-23] : 최종 체크리스트
# [핵심] 프로젝트 완성도 자가 점검
# [주의] 모든 항목이 True여야 시연 준비 완료
checklist = {
    "엑셀 Agent 구성":     "excel_agent" in dir(),
    "SQL Agent 구성":      "sql_agent"   in dir(),
    "RAG Agent 구성":      "rag_agent"   in dir(),
    "SKILL.md 정의":       "EXCEL_SKILL" in dir() and "SQL_SKILL" in dir() and "RAG_SKILL" in dir(),
    "SubAgent 등록":       "subagents"   in dir(),
    "Supervisor 컴파일":   "app"         in dir(),
    "라우팅 테스트 완료":   True,   # 위 테스트 실행 후 True 확인
}

all_pass = all(checklist.values())
print("=== 최종 체크리스트 ===\n")
for item, status in checklist.items():
    icon = "[v]" if status else "[ ]"
    print(f"  {icon} {item}")
print(f"\n{'모든 항목 통과' if all_pass else '미완료 항목 확인 필요'}")


=== 최종 체크리스트 ===

  [v] 엑셀 Agent 구성
  [v] SQL Agent 구성
  [v] RAG Agent 구성
  [v] SKILL.md 정의
  [v] SubAgent 등록
  [v] Supervisor 컴파일
  [v] 라우팅 테스트 완료

모든 항목 통과


---

## 정리

| 항목 | 내용 |
|------|------|
| **아키텍처** | SKILL.md 기반 SubAgent + Supervisor 오케스트레이션 |
| **SKILL.md** | 각 Agent의 행동·도구를 선언적으로 정의하는 파일 |
| **SubAgent** | 독립된 컨텍스트에서 실행 — 컨텍스트 블로트 방지 |
| **엑셀 SubAgent** | SKILL.md + pandas Tool — 예산·진행률 분석 |
| **SQL SubAgent** | SKILL.md + SQLite Tool — 태스크·리소스 조회 |
| **RAG SubAgent** | SKILL.md + 인메모리 벡터스토어 검색 — 절차·정책 문서 검색 |
| **Supervisor** | `create_deep_agent(subagents=...)` — SubAgent 기반 라우팅 |
| **핵심 기술** | W1(엑셀) + W2(SQL) + W3(RAG) + SKILL.md + SubAgent 통합 |

### 확장 방향

- **새 Agent 추가**: SKILL.md 파일 하나 + SubAgent dict 추가로 즉시 확장
- **SKILL.md 재사용**: 다른 프로젝트에서도 동일 SKILL.md로 Agent 행동 재현
- **컨텍스트 격리**: SubAgent별 독립 컨텍스트로 대규모 멀티에이전트 안정 운영
- LangSmith/Langfuse 트레이싱으로 SubAgent 위임 품질 모니터링
- Docker + langgraph serve로 프로덕션 배포
